# Bygge grafen

Målet er å skape en graf som
- Er fri for skala, og
- som også holder våre kriminelle.
Det vil i praksis si å lese inn grafen, og så legge til nodene (og deres relasjoner) slik vi har forberedt.

For at våre kriminelle skal gli naturlig inn, må de gis relasjoner til de eksisterende på en "naturlig" måte.  Det vil si at hver node må få (et antall) relasjoner tli andre noder som "ligner".  I praksis betyr det å ha større sjande for å knytte seg til noder som alerede har mange kanter.

Nå har (noen) av våre kriminelle trolig flere kanter enn medianen i datasettet, men det kan vi se på senere.

Eneste måten jeg har funnet for å legge inn ny noder på en "normal" måte, er å lage et sett av alle noder, hvor hver node er i settet like mange ganger som det har kanter.  Når én node nå trekkes fra settet vil det sannsynligheten for å trekke en node reflektere nodens sentralitet.





In [4]:
# Laste inn datasettet
# La oss lage en graf
import networkx as nx
import gzip

EG = nx.DiGraph()
# husk at gzip åpner i 'b'
with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    for linje in fd:
         link = linje.split()
         EG.add_edge(int(link[0]), int(link[1]))
    #
#

# Bort med eposter sendt til seg selv
EG.remove_edges_from(nx.selfloop_edges(EG))
isolerte = list(nx.isolates(EG)) # kan ikke bruke iteratorer direkte
EG.remove_nodes_from(isolerte)

# Merke nodene slik at det blir riktig ved import i neo4j senere
# Dette er APOC-kompatibel måte å gjøre det på
nx.set_node_attributes(EG, ":Person", name="label")
# Og kantene
nx.set_edge_attributes(EG, True, name="Epost")

print(EG)

DiGraph with 57189 nodes and 103083 edges


In [6]:
# Hvor mange deler består grafen av.
# Vi bryr oss ikke om retningen for om A->B->C så er A knyttet til C
print(f"Grafen består av {nx.number_weakly_connected_components(EG)} deler")

deler = list(nx.weakly_connected_components(EG))
# Hent ut antall 
antall = {}
for n, noder in enumerate(deler):
    antall[n] = len(noder)
#
print(antall)
# Ett seksmpel
print(f"Ett eksempel (1): {deler[1]}")

Grafen består av 185 deler
{0: 56576, 1: 5, 2: 8, 3: 2, 4: 3, 5: 4, 6: 4, 7: 2, 8: 4, 9: 8, 10: 2, 11: 3, 12: 2, 13: 2, 14: 2, 15: 3, 16: 6, 17: 3, 18: 3, 19: 6, 20: 2, 21: 2, 22: 2, 23: 2, 24: 3, 25: 2, 26: 2, 27: 2, 28: 3, 29: 2, 30: 3, 31: 2, 32: 2, 33: 3, 34: 3, 35: 2, 36: 2, 37: 2, 38: 2, 39: 3, 40: 2, 41: 2, 42: 4, 43: 2, 44: 2, 45: 2, 46: 2, 47: 7, 48: 146, 49: 2, 50: 2, 51: 2, 52: 2, 53: 3, 54: 2, 55: 3, 56: 2, 57: 3, 58: 2, 59: 3, 60: 2, 61: 2, 62: 2, 63: 2, 64: 2, 65: 4, 66: 4, 67: 4, 68: 3, 69: 4, 70: 3, 71: 3, 72: 3, 73: 2, 74: 3, 75: 2, 76: 2, 77: 2, 78: 2, 79: 2, 80: 3, 81: 2, 82: 2, 83: 2, 84: 2, 85: 3, 86: 3, 87: 4, 88: 2, 89: 2, 90: 2, 91: 4, 92: 2, 93: 2, 94: 11, 95: 5, 96: 2, 97: 2, 98: 2, 99: 4, 100: 3, 101: 2, 102: 2, 103: 2, 104: 2, 105: 2, 106: 2, 107: 2, 108: 2, 109: 2, 110: 2, 111: 2, 112: 3, 113: 2, 114: 2, 115: 3, 116: 2, 117: 2, 118: 2, 119: 2, 120: 2, 121: 3, 122: 2, 123: 2, 124: 3, 125: 2, 126: 2, 127: 2, 128: 2, 129: 2, 130: 2, 131: 2, 132: 3, 133: 2, 134

In [7]:
# Fjerne alle de små
største = max(nx.weakly_connected_components(EG), key=len)
EPOST = EG.subgraph(største).copy()
print(EPOST)

DiGraph with 56576 nodes and 102631 edges


Vi skal legge til den "økonomiske kriminaliteten" som vi har laget.  Det må vi gjøre på en "naturlig" måte.  De nye nodene, i tillegg til sine egenskaper, må "gli inn" i landskapet på en naturlig måte.  Vi følger Barabási i (Kapittel 5.2 om *preferential attachment*)[https://networksciencebook.com/chapter/5#growth] og knytter våre noder til eksisterende slik at hvilken node vi skal knytte noder til avhenger av nodens sentralitet.

Vi bygger en liste, hvor antall ganger en node er i listen er lik antall kanter noden har.  Det gjør at når vi skal velge en node tilfeldig, er sjansen størst for at vi knytter oss til sentrale noder (i tråd med teorien).

Jeg spør Gemini:
```
Using networkx, without converting the whole graph to a list, how can I find a nrandom node
```
Den svarer
```
random_node = random.sample(G.nodes, 1)[0]
```
Men når jeg fortsetter:
```
Are you sure?  In your code "random_node = random.sample(G.nodes, 1)[0]" seems to create a list before returning one element
```
Svaret er
```
To be intellectually honest: internally, it still performs an $O(n)$ operation, [...]
```
Eller, som alltid: Livet er lettere når man vet svaret.  Forøvrig er det et interessant spørsmål hva *intellectually honest* skal bety (i motsetning til "bare" *honest* mener jeg).

In [24]:
# Bygg listen
alle_noder = []
for u, v in EPOST.edges():
    # Hver kant gir to noder.
    alle_noder.extend([u,v])
#
print(f"Antall noder i listen: {len(alle_noder)}")

Antall noder i listen: 205262


In [16]:
# Laste inn bakmann
Bakmann = nx.read_graphml("grafer/Bakmann.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Bakmann, True, name="Bakmann")
# Sjekke at det ser bra ut  
print(Bakmann)


Graph with 24 nodes and 49 edges


In [17]:
# Laste inn "money mule"
Esel = nx.read_graphml("grafer/Mule10.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Esel, True, name="Esel")
# Verify by checking the number of nodes and edges
print(Esel)


Graph with 44 nodes and 119 edges


In [18]:
# Laste inn deling av utbytte
Utbytte = nx.read_graphml("grafer/Utbytte.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Utbytte, True, name="Utbytte")
# Verify by checking the number of nodes and edges
print(Utbytte)

Graph with 81 nodes and 159 edges


`EPOST` er en rettet graf.  Men dersom A -> B så er jo B knytte til A slik at det er like riktig å snakke om A -- B som "kun" A -> B.  Vi gjør livet vårt litt enklere, og konverterer til en urettet graf.

In [33]:
GRAF = EPOST.to_undirected()
print(EPOST)
print(GRAF)

DiGraph with 56576 nodes and 102631 edges
Graph with 56576 nodes and 92013 edges


In [ ]:
import random
print(f"Originalen: {GRAF}")
# Legg de små grafene til i den store
Komplett = GRAF
for g in Bakmann, Esel, Utbytte:
    # Nodene i de små grafene har identifikatorer som er UUID, så ingen er like
    # Vi kan derfor trygt slå dem sammen
    Komplett = nx.union(Komplett, g)
    for n in g:
        tilfeldig = random.choice(alle_noder)
        # Hent en tilfeldig valgt node fra den originale grafen
        Komplett.add_edge(n, tilfeldig)
#
print(f"Den komplette: {Komplett}")
print(f"Grafen består av {nx.number_connected_components(Komplett)} del")

Originalen: Graph with 56576 nodes and 92013 edges
Den komplette: Graph with 56725 nodes and 92489 edges
Grafen består av 1 dele


In [32]:
# Lagre grafen
nx.write_graphml(Komplett, "grafer/Komplett.graphml")